# Standard and double-interface VASP workflows

Prepare two complete MLIP → VASP workflows:

- Si/Ge single interface: bulk static, interface/slab relax → static, and strain correction.
- MgO/MgO double interface: bulk static and interface relax → static.

The tested server used VASP 6.5.1. Update the partition and module command for your cluster.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import InterOptimus

candidates = (Path.cwd(), Path.cwd().parent, Path(InterOptimus.__file__).resolve().parents[1])
REPO_ROOT = next((path for path in candidates if (path / "scripts").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the InterOptimus repository or its examples directory")
sys.path.insert(0, str(REPO_ROOT))

from InterOptimus.agents.simple_iomaker import run_simple_iomaker
from scripts.prepare_interface_workflow_validation import structures, workflow

WORKDIR = Path.cwd() / "remote_vasp_standard_examples"
SUBMIT = False
VASP_PARTITION = "spr"  # The validation fallback used "gpu2" while spr was saturated.
WORKDIR.mkdir(exist_ok=True)
(WORKDIR / "materials").mkdir(exist_ok=True)
(WORKDIR / "configs").mkdir(exist_ok=True)

materials = structures()
for name in ("si", "ge", "mgo"):
    materials[name].to(filename=WORKDIR / "materials" / f"{name}.cif")

configs = [
    workflow(name="example_vasp_standard_si_ge", film="si", substrate="ge", calc="orb-models", worker="orb", miller=(1, 1, 1), max_area=15, strain=True, do_vasp=True),
    workflow(name="example_vasp_double_mgo", film="mgo", substrate="mgo", calc="matris", worker="matris", miller=(0, 0, 1), max_area=20, double_interface=True, do_vasp=True),
]
for config in configs:
    bulk = config["IO_workflow_config"]["bulk_cifs"]
    bulk["film_cif"] = str((WORKDIR / bulk["film_cif"]).resolve())
    bulk["substrate_cif"] = str((WORKDIR / bulk["substrate_cif"]).resolve())
    config["cluster"]["vasp"]["vasp_slurm_partition"] = VASP_PARTITION
    path = WORKDIR / "configs" / f"{config['workflow_name']}.json"
    path.write_text(json.dumps(config, indent=2))

[(config["workflow_name"], config["cluster"]["vasp"]["vasp_slurm_partition"]) for config in configs]

In [ ]:
results = {}
if SUBMIT:
    for config in configs:
        result = run_simple_iomaker(config)
        results[config["workflow_name"]] = result
        print(config["workflow_name"], result["mlip_job_uuid"])
else:
    print("Configuration only. Confirm POTCAR, VASP module, workers, and partition before submission.")

results